In [1]:
import os, certifi
os.environ["SSL_CERT_FILE"] = certifi.where()

from pathlib import Path


import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

DATA_DIR = Path("data/atp")
OUT_DIR = Path("outputs")
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

YEARS = range(2015, 2027)
BASE_URL = BASE_URL = "https://raw.githubusercontent.com/Kadantte/tennis_atp/master/atp_matches_{year}.csv"

TRAIN_END_YEAR = 2022   # train: year <= this, test: year > this
FORM_WEEKS = 8          # window for "recent form"
BASELINE_WEEKS = 52     # window for the player's own baseline
MIN_FORM_POINTS = 150   # min points played in the window to trust the form estimate
EPOCH = pd.Timestamp("2000-01-03")  # a Monday; week_idx counts weeks since this


def year_of(week_idx):
    return (EPOCH + pd.to_timedelta(week_idx * 7, unit="D")).dt.year


def report(name, y, p, bins=5):
    y, p = np.asarray(y), np.asarray(p)
    print(
        f"{name}: n={len(y):,} base_rate={y.mean():.4f} "
        f"logloss={log_loss(y, p):.4f} brier={brier_score_loss(y, p):.4f} "
        f"auc={roc_auc_score(y, p):.3f}"
    )
    cal = pd.DataFrame({"p": p, "y": y})
    cal["bin"] = pd.qcut(cal["p"], bins, duplicates="drop")
    print(
        cal.groupby("bin", observed=True)
        .agg(pred=("p", "mean"), obs=("y", "mean"), n=("y", "size"))
        .round(4)
    )

## 1. Load match data (cached locally after first download)

In [2]:
import io
import ssl
import urllib.request

import certifi

SSL_CTX = ssl.create_default_context(cafile=certifi.where())


def fetch_csv(url):
    with urllib.request.urlopen(url, context=SSL_CTX, timeout=30) as resp:
        return pd.read_csv(io.BytesIO(resp.read()))


def load_year(year):
    path = DATA_DIR / f"atp_matches_{year}.csv"
    if not path.exists():
        try:
            fetch_csv(BASE_URL.format(year=year)).to_csv(path, index=False)
        except Exception as exc:
            print(f"  {year}: FAILED -> {exc!r}")
            return None
    return pd.read_csv(path)


frames = [f for f in (load_year(y) for y in YEARS) if f is not None]
if not frames:
    raise RuntimeError("No match files loaded. See the per-year errors above.")
matches = pd.concat(frames, ignore_index=True)

matches = matches.dropna(subset=["tourney_date", "winner_id", "loser_id"]).copy()

matches["date"] = pd.to_datetime(
    matches["tourney_date"].astype(int).astype(str), format="%Y%m%d"
)
matches["week_idx"] = ((matches["date"] - EPOCH).dt.days // 7).astype(int)

score = matches["score"].fillna("")
matches["is_wo"] = score.str.contains("W/O", regex=False)       # walkover: no match played
matches["is_end"] = score.str.contains("RET|DEF", regex=True)   # retired / defaulted mid-match

matches["minutes"] = pd.to_numeric(matches["minutes"], errors="coerce")
matches["minutes"] = matches["minutes"].fillna(
    matches.groupby("best_of")["minutes"].transform("median")
)
matches.loc[matches["is_wo"], "minutes"] = 0
matches["minutes"] = matches["minutes"].fillna(0)

print(f"{len(matches):,} matches, {matches['date'].min().date()} to {matches['date'].max().date()}")

32,065 matches, 2015-01-04 to 2026-05-25


## 2. Player-match rows (one row per player per match)
Serve points won, return points won (derived from the opponent's serve stats), minutes,
and injury proxies (the loser of a RET/DEF match is the player who stopped).

In [3]:
def one_side(df, me, opp):
    m, o = me[0], opp[0]  # 'w' / 'l'
    is_loser = me == "loser"
    return pd.DataFrame(
        {
            "player_id": df[f"{me}_id"],
            "opp_id": df[f"{opp}_id"],
            "week_idx": df["week_idx"],
            "won": int(me == "winner"),
            "played": ~df["is_wo"],
            "minutes": df["minutes"],
            "age": df[f"{me}_age"],
            "opp_age": df[f"{opp}_age"],
            "rank": df[f"{me}_rank"],
            "opp_rank": df[f"{opp}_rank"],
            "sp_won": df[f"{m}_1stWon"] + df[f"{m}_2ndWon"],
            "sp_played": df[f"{m}_svpt"],
            "rp_won": df[f"{o}_svpt"] - df[f"{o}_1stWon"] - df[f"{o}_2ndWon"],
            "rp_played": df[f"{o}_svpt"],
            "inj": (df["is_end"] & is_loser).astype(int),
            "wo": (df["is_wo"] & is_loser).astype(int),
        }
    )


pm = pd.concat(
    [one_side(matches, "winner", "loser"), one_side(matches, "loser", "winner")],
    ignore_index=True,
)
pm["played"] = pm["played"].astype(int)
for c in ["sp_won", "sp_played", "rp_won", "rp_played"]:
    pm[c] = pm[c].fillna(0)
pm.shape

(64130, 16)

## 3. Player-week panel with workload and form features
Complete weekly calendar per player (rest weeks are explicit rows with zeros).
Features at row `t` use information through the END of week `t` (the state at the start of `t+1`).

In [4]:
pw = (
    pm.groupby(["player_id", "week_idx"])
    .agg(
        matches=("played", "sum"),
        minutes=("minutes", "sum"),
        sp_won=("sp_won", "sum"),
        sp_played=("sp_played", "sum"),
        rp_won=("rp_won", "sum"),
        rp_played=("rp_played", "sum"),
        inj=("inj", "max"),
        wo=("wo", "max"),
        age=("age", "last"),
        rank=("rank", "last"),
    )
    .reset_index()
)

bounds = pw.groupby("player_id")["week_idx"].agg(["min", "max"])
lengths = (bounds["max"] - bounds["min"] + 1).to_numpy()
grid = pd.DataFrame(
    {
        "player_id": np.repeat(bounds.index.to_numpy(), lengths),
        "week_idx": np.concatenate(
            [np.arange(a, b + 1) for a, b in zip(bounds["min"], bounds["max"])]
        ),
    }
)
grid = grid.merge(pw, on=["player_id", "week_idx"], how="left")
count_cols = ["matches", "minutes", "sp_won", "sp_played", "rp_won", "rp_played", "inj", "wo"]
grid[count_cols] = grid[count_cols].fillna(0)
grid = grid.sort_values(["player_id", "week_idx"]).reset_index(drop=True)
grid[["age", "rank"]] = grid.groupby("player_id")[["age", "rank"]].ffill()
grid["year"] = year_of(grid["week_idx"])


def roll_sum(col, w):
    """Sum over the current week and the previous w-1 calendar weeks."""
    cs = grid.groupby("player_id")[col].cumsum()
    return cs - cs.groupby(grid["player_id"]).shift(w).fillna(0)


# workload
grid["min_1w"] = grid["minutes"]
grid["min_4w"] = roll_sum("minutes", 4)
grid["min_12w"] = roll_sum("minutes", 12)

last_played = grid["week_idx"].where(grid["matches"] > 0)
grid["last_played"] = last_played.groupby(grid["player_id"]).ffill()
grid["weeks_since"] = grid["week_idx"] - grid["last_played"]  # 0 if played this week

# form = recent points-won % minus the player's own 52-week baseline
for w, tag in [(FORM_WEEKS, "f"), (BASELINE_WEEKS, "b")]:
    won = roll_sum("sp_won", w) + roll_sum("rp_won", w)
    played = roll_sum("sp_played", w) + roll_sum("rp_played", w)
    grid[f"pct_{tag}"] = (won / played).where(played >= MIN_FORM_POINTS)
grid["form_delta"] = grid["pct_f"] - grid["pct_b"]

print(f"{len(grid):,} player-weeks, {grid['player_id'].nunique():,} players")
grid[["form_delta", "min_4w", "weeks_since"]].describe().round(3)

233,884 player-weeks, 1,331 players


,form_delta,min_4w,weeks_since
count,83169.000,233884.000,233884.000
mean,-0.003,120.995,32.283
std,0.021,239.578,53.574
min,-0.156,0.000,0.000
25%,-0.012,0.000,2.000
50%,0.000,0.000,11.000
75%,0.008,143.000,38.000
max,0.146,2308.000,469.000


In [8]:
## Ok let's view our data
pd.set_option("display.max_columns", None)

print(grid.shape)
grid.head(50)



(233884, 21)


,player_id,week_idx,matches,minutes,sp_won,sp_played,rp_won,rp_played,inj,wo,age,rank,year,min_1w,min_4w,min_12w,last_played,weeks_since,pct_f,pct_b,form_delta
0,100644,788,1.0,66.0,26.0,51.0,9.0,45.0,0.0,0.0,17.8,137.0,2015,66.0,66.0,66.0,788.0,0.0,NaN,NaN,NaN
1,100644,789,1.0,86.0,39.0,69.0,17.0,54.0,0.0,0.0,17.8,138.0,2015,86.0,152.0,152.0,789.0,0.0,0.415525,0.415525,0.000000
2,100644,790,1.0,76.0,33.0,54.0,20.0,66.0,0.0,0.0,17.8,136.0,2015,76.0,228.0,228.0,790.0,0.0,0.424779,0.424779,0.000000
3,100644,791,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,17.8,136.0,2015,0.0,228.0,228.0,790.0,1.0,0.424779,0.424779,0.000000
4,100644,792,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,17.8,136.0,2015,0.0,162.0,228.0,790.0,2.0,0.424779,0.424779,0.000000
5,100644,793,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,17.8,136.0,2015,0.0,76.0,228.0,790.0,3.0,0.424779,0.424779,0.000000
6,100644,794,2.0,213.0,121.0,177.0,36.0,156.0,0.0,0.0,17.9,129.0,2015,213.0,213.0,441.0,794.0,0.0,0.447917,0.447917,0.000000
7,100644,795,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,17.9,129.0,2015,0.0,213.0,441.0,794.0,1.0,0.447917,0.447917,0.000000
8,100644,796,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,17.9,129.0,2015,0.0,213.0,441.0,794.0,2.0,0.461806,0.447917,0.013889
9,100644,797,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,17.9,129.0,2015,0.0,213.0,441.0,794.0,3.0,0.463576,0.447917,0.015659


In [9]:
# Modeling injury rate
# Since defaulting a match is pretty rare, let's make it non-player specific and try to fit a curve

# Injury-proxy table: consecutive weeks played x age group
# Event at the end of a played week = retired mid-match that week
#                                     OR more than OFF_THRESHOLD empty weeks before the next tour-level match.
OFF_THRESHOLD = 5        # "more than 3 weeks off" -> at least 4 empty weeks before the next match
STREAK_CAP = 6           # streaks of 4 or more weeks are pooled into "4+"
EXCLUDE_YEARS = (2020,)  # COVID suspension would look like a mass injury

gr = grid.sort_values(["player_id", "week_idx"]).copy()
is_played = gr["matches"] > 0

# consecutive weeks played, ending in this week (0 on rest weeks)
run_id = (~is_played).groupby(gr["player_id"]).cumsum()
gr["streak"] = is_played.groupby([gr["player_id"], run_id]).cumsum()

wk = gr[is_played].copy()
by_player = wk.groupby("player_id")
wk["next_week"] = by_player["week_idx"].shift(-1)
wk["next_year"] = by_player["year"].shift(-1)
wk["weeks_off"] = wk["next_week"] - wk["week_idx"] - 1

wk["retired"] = wk["inj"] > 0
# gaps that cross New Year are treated as off-season, not injury
wk["long_off"] = (wk["weeks_off"] > OFF_THRESHOLD) & (wk["next_year"] == wk["year"])
wk["event"] = wk["retired"] | wk["long_off"]

# keep only rows where the outcome is fully observable
last_week = gr["week_idx"].max()
n_before = len(wk)
keep = (
    wk["age"].notna()
    & ~wk["year"].isin(EXCLUDE_YEARS)
    & (wk["week_idx"] <= last_week - OFF_THRESHOLD - 1)
)
wk = wk[keep].copy()
print(f"played weeks: {n_before:,} -> kept {len(wk):,} "
      f"(dropped missing age, excluded years {EXCLUDE_YEARS}, and the last {OFF_THRESHOLD + 1} weeks of data)")

# 3 age groups: terciles over all kept played weeks
age_cuts = wk["age"].quantile([1 / 3, 2 / 3]).to_numpy()
age_labels = [f"<={age_cuts[0]:.1f}", f"{age_cuts[0]:.1f}-{age_cuts[1]:.1f}", f">{age_cuts[1]:.1f}"]
wk["age_group"] = pd.cut(wk["age"], [-np.inf, *age_cuts, np.inf], labels=age_labels)
print("age cut points:", age_cuts.round(2))

streak_labels = [str(i) for i in range(1, STREAK_CAP)] + [f"{STREAK_CAP}+"]
wk["streak_bucket"] = pd.Categorical(
    np.where(wk["streak"] >= STREAK_CAP, f"{STREAK_CAP}+", wk["streak"].astype(int).astype(str)),
    categories=streak_labels,
    ordered=True,
)


def wilson(k, n, z=1.96):
    """95% Wilson score interval for a proportion k/n."""
    phat = k / n
    denom = 1 + z**2 / n
    centre = (phat + z**2 / (2 * n)) / denom
    half = z * np.sqrt(phat * (1 - phat) / n + z**2 / (4 * n**2)) / denom
    return centre - half, centre + half


tab = (
    wk.groupby(["age_group", "streak_bucket"], observed=True)
    .agg(n=("event", "size"), retired=("retired", "sum"), long_off=("long_off", "sum"), events=("event", "sum"))
    .reset_index()
)
tab["rate"] = tab["events"] / tab["n"]
tab["ci_low"], tab["ci_high"] = wilson(tab["events"], tab["n"])
tab["rate_retired"] = tab["retired"] / tab["n"]
tab["rate_long_off"] = tab["long_off"] / tab["n"]

print(f"overall: event rate {wk['event'].mean():.4f} "
      f"(retired {wk['retired'].mean():.4f}, long off {wk['long_off'].mean():.4f})")

# readable version: for the overall event and each of its two parts,
# show count / n, the rate, and the Wilson interval (numeric table stays in `tab`)
def fmt_block(df, k_col, label):
    k, n = df[k_col], df["n"]
    lo, hi = wilson(k, n)
    out = pd.DataFrame({
        "count / n": k.astype(int).astype(str) + " / " + n.map("{:,}".format),
        "rate": (k / n * 100).map("{:.1f}%".format),
        "95% CI": (lo * 100).map("{:.1f}%".format) + " to " + (hi * 100).map("{:.1f}%".format),
    })
    out.columns = pd.MultiIndex.from_product([[label], out.columns])
    return out


show = pd.concat(
    [
        fmt_block(tab, "events", "Any event"),
        fmt_block(tab, "retired", "Retired"),
        fmt_block(tab, "long_off", "Long off"),
    ],
    axis=1,
)
show.index = pd.MultiIndex.from_frame(
    tab[["age_group", "streak_bucket"]].astype(str), names=["Age group", "Consecutive weeks"]
)

# rendered table: grouped headers, centered cells, a divider between the three blocks
cell_css = [("text-align", "center"), ("padding", "6px 12px")]
divider = [("border-left", "2px solid gray")]
styles = [
    {"selector": "th", "props": cell_css},
    {"selector": "td", "props": cell_css},
    {"selector": "caption", "props": [("caption-side", "top"), ("text-align", "left"),
                                      ("font-weight", "bold"), ("padding-bottom", "8px")]},
]
for c in (3, 6):  # first column of the Retired and Long off blocks
    styles.append({"selector": f"td.col{c}", "props": divider})
    styles.append({"selector": f"th.col_heading.col{c}", "props": divider})

styled = (
    show.style
    .set_caption("Injury proxy (retirement or more than 3 weeks off) by age group and consecutive weeks played")
    .set_table_styles(styles)
    .set_properties(subset=pd.IndexSlice[:, [("Any event", "rate")]], **{"font-weight": "bold"})
)
display(styled)

by_age = tab.groupby("age_group", observed=True)[["retired", "n"]].sum()
by_age["retired_rate"] = by_age["retired"] / by_age["n"]
by_age["ci_low"], by_age["ci_high"] = wilson(by_age["retired"], by_age["n"])
by_age
# Notice how the retired rate does not differ substatially between age groups 

played weeks: 33,266 -> kept 31,359 (dropped missing age, excluded years (2020,), and the last 6 weeks of data)
age cut points: [25.  29.1]
overall: event rate 0.1017 (retired 0.0269, long off 0.0782)


,retired,n,retired_rate,ci_low,ci_high
age_group,,,,,
<=25.0,268,10537,0.025434,0.022597,0.028618
25.0-29.1,265,10554,0.025109,0.022292,0.028271
>29.1,312,10268,0.030386,0.027237,0.033885


In [ ]:
# Physical level modeling
grid.head()
# Perhaps, just use the pct_f and pct_b to measure a physical level 
# K clustering algorithm to group tiers for players and map that to win percentage

,player_id,week_idx,matches,minutes,sp_won,sp_played,rp_won,rp_played,inj,wo,age,rank,year,min_1w,min_4w,min_12w,last_played,weeks_since,pct_f,pct_b,form_delta
0,100644,788,1.0,66.0,26.0,51.0,9.0,45.0,0.0,0.0,17.8,137.0,2015,66.0,66.0,66.0,788.0,0.0,NaN,NaN,NaN
1,100644,789,1.0,86.0,39.0,69.0,17.0,54.0,0.0,0.0,17.8,138.0,2015,86.0,152.0,152.0,789.0,0.0,0.415525,0.415525,0.0
2,100644,790,1.0,76.0,33.0,54.0,20.0,66.0,0.0,0.0,17.8,136.0,2015,76.0,228.0,228.0,790.0,0.0,0.424779,0.424779,0.0
3,100644,791,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,17.8,136.0,2015,0.0,228.0,228.0,790.0,1.0,0.424779,0.424779,0.0
4,100644,792,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,17.8,136.0,2015,0.0,162.0,228.0,790.0,2.0,0.424779,0.424779,0.0
